# **A1-ÖNİŞLEME**

### **İş akışı özet:**

- **Uydu Jeodezisi 1 ödevinde ISTA noktası gözlem (14o), meteoroloji (14m), navigasyon (14n), ve final orbit (sp3) dosyaları verildi. Bunlar data klasörüne gaydedildi. Data klasörü gitignore içinde githuba yüklenmedi.**
- **Data klasöründeki file path olarak alınıp içinden verilen satırlar arası bilgiyi kaydeden fonksiyon yazıldı ve 2014  8 28 17 15 00.0000 epoğundaki veriler kaydedildi.**
- **İlk satırdaki epok ve uydu isimleri ayrıca epoch ve satellites olarak kaydedildi. 1-VERİLERİ OKUMA VE KAYDETME kısmı bitti.**
- **Epokta görünen 9 uydu ismi ve verileri ayrı ayrı sat1-sat9 arası listelere kaydedildi. 2-VERİLERİ LİSTELERE AYIR kısmı bitti. <span style="color:red">Fazlar eksi kaldı, abs() içine alınmalımı?</span>**
- **Veriler dışarıya satellites.csv olarak kaydedildi. 3-DIŞARIYA AKTAR kısmı bitti.**

## **1-VERİLERİ OKUMA VE KAYDETME**

- **Bazı önemli değişkenler:**
- **obs_file = ISTA2400.14o**
- **data = 4  8 28 17 15  0.0000000 bu epoktaki veriler**
- **epoch = epok**
- **sat_list = uydu listesi**
- **obs_data = gözlem verileri**

In [1]:
from pathlib import Path

In [2]:
data_dir = Path("../data")

In [3]:
obs_file = data_dir / "ISTA2400.14o"
nav_file = data_dir / "ISTA2400.14n"
sp3_file = data_dir / "igs18074.sp3"

In [4]:
# başlangıç ile bitiş satırı arası verileri toplar, satırın tamamını girdi olarak yaz, boşluklar ile beraber kopyala
def get_epoch_data(file_path, start_epoch, end_epoch):
    data = []
    collecting = False

    with open(file_path, "r") as f:
        for line in f:
            if line.rstrip() == start_epoch:
                collecting = True

            if line.rstrip() == end_epoch:
                break

            if collecting:
                data.append(line.rstrip())

    return data

In [5]:
#14  8 28 17 15  0 epoğu verileri toplandı data değişkeninde.
data = get_epoch_data(
    obs_file,
    " 14  8 28 17 15  0.0000000  0  9G19G27G11G20G22G14G32G 1G31",
    " 14  8 28 17 15 30.0000000  0  9G19G27G11G20G22G14G32G 1G31"
)

In [6]:
# C1    P1    L1    D1    P2    L2    D2
data

[' 14  8 28 17 15  0.0000000  0  9G19G27G11G20G22G14G32G 1G31',
 '  21189058.919    21189058.224   -23422371.33509     -1624.826    21189062.761',
 ' -18232789.09909     -1266.095',
 '  22520607.109    22520607.184   -16265062.30809     -3131.943    22520615.610',
 ' -12664483.16409     -2440.473',
 '  20960433.392    20960432.985   -20236898.56309      2125.069    20960438.054',
 ' -15753030.25009      1655.895',
 '  24599610.601    24599611.385    -4001554.05609      3287.044    24599619.779',
 '  -2858090.76509      2561.330',
 '  22343853.992    22343853.460   -19008308.32409     -2218.107    22343856.607',
 ' -14786004.90809     -1728.398',
 '  21370295.018    21370294.498   -19001406.30909      -529.566    21370299.873',
 ' -14553929.34709      -412.656',
 '  21128427.864    21128427.325   -19895704.40009      1824.349    21128433.745',
 ' -15488787.19409      1421.562',
 '  22556048.986    22556050.480   -15143902.88409      2922.988    22556058.525',
 ' -11776649.86109      227

In [7]:
# ilk satır metadata ayrıştırılıcak
first_row = data[0]
first_row

' 14  8 28 17 15  0.0000000  0  9G19G27G11G20G22G14G32G 1G31'

In [8]:
# ortadaki 0 epok flag 0=normal gözlem, 1=power failure sonrası vs. onu yok saydım.
epoch = first_row[:26]

In [9]:
# uydu sayısını ifade eden 9 sayısınıda yok saydım.
satellites = first_row[32:]

In [10]:
print(epoch)
print(satellites)

 14  8 28 17 15  0.0000000
G19G27G11G20G22G14G32G 1G31


In [11]:
# sat_list verilen uyduların prnlerini kaydet
sat_list = []
for i in range(0, len(satellites), 3):
    sat_list.append(satellites[i:i+3])

In [12]:
# 0 yazılmamış düzeltildi
sat_list[7] = "G01"

In [13]:
print(sat_list)

['G19', 'G27', 'G11', 'G20', 'G22', 'G14', 'G32', 'G01', 'G31']


In [14]:
# obs_data gözlemler epok-uydu satırı hariç seçildi
obs_data = data[1:]

In [15]:
obs_data

['  21189058.919    21189058.224   -23422371.33509     -1624.826    21189062.761',
 ' -18232789.09909     -1266.095',
 '  22520607.109    22520607.184   -16265062.30809     -3131.943    22520615.610',
 ' -12664483.16409     -2440.473',
 '  20960433.392    20960432.985   -20236898.56309      2125.069    20960438.054',
 ' -15753030.25009      1655.895',
 '  24599610.601    24599611.385    -4001554.05609      3287.044    24599619.779',
 '  -2858090.76509      2561.330',
 '  22343853.992    22343853.460   -19008308.32409     -2218.107    22343856.607',
 ' -14786004.90809     -1728.398',
 '  21370295.018    21370294.498   -19001406.30909      -529.566    21370299.873',
 ' -14553929.34709      -412.656',
 '  21128427.864    21128427.325   -19895704.40009      1824.349    21128433.745',
 ' -15488787.19409      1421.562',
 '  22556048.986    22556050.480   -15143902.88409      2922.988    22556058.525',
 ' -11776649.86109      2277.661',
 '         0.000    25209285.799      -87471.64109      

## **2-VERİLERİ LİSTELERE AYIR**

- **sat1 den sat 9a kadar liste oluşturuldu.**
- **uydu prn sonrası C1    P1    L1    D1    P2    L2    D2 şeklinde metadata gibi kaydedildi**
- **lli ssi ayıklandı**
- **veriler float formatına çevrildi**

In [16]:
# sat1 den sat 9 a kadar liste oluştur ve listelere ilk element olarak uydu prn koy
for i in range(len(sat_list)):
    globals()[f"sat{i+1}"] = []
    globals()[f"sat{i+1}"].append(sat_list[i])
    print(globals()[f"sat{i+1}"])

['G19']
['G27']
['G11']
['G20']
['G22']
['G14']
['G32']
['G01']
['G31']


In [17]:
# listelere RINEX ilk satırı koy
for i in range(9):
    line = obs_data[2*i]
    data1 = line[2:14]
    data2 = line[18:30]
    data3 = line[33:48]
    data4 = line[53:62]
    data5 = line[66:78]
    globals()[f"sat{i+1}"].append(data1)
    globals()[f"sat{i+1}"].append(data2)
    globals()[f"sat{i+1}"].append(data3)
    globals()[f"sat{i+1}"].append(data4)
    globals()[f"sat{i+1}"].append(data5)
    print(globals()[f"sat{i+1}"])

['G19', '21189058.919', '21189058.224', '-23422371.33509', '-1624.826', '21189062.761']
['G27', '22520607.109', '22520607.184', '-16265062.30809', '-3131.943', '22520615.610']
['G11', '20960433.392', '20960432.985', '-20236898.56309', ' 2125.069', '20960438.054']
['G20', '24599610.601', '24599611.385', ' -4001554.05609', ' 3287.044', '24599619.779']
['G22', '22343853.992', '22343853.460', '-19008308.32409', '-2218.107', '22343856.607']
['G14', '21370295.018', '21370294.498', '-19001406.30909', ' -529.566', '21370299.873']
['G32', '21128427.864', '21128427.325', '-19895704.40009', ' 1824.349', '21128433.745']
['G01', '22556048.986', '22556050.480', '-15143902.88409', ' 2922.988', '22556058.525']
['G31', '       0.000', '25209285.799', '   -87471.64109', ' 3119.319', '25220575.865']


In [18]:
# listelere RINEX ikinci satırı koy
for i in range(9):
    line = obs_data[2*i+1]
    data1 = line[1:16]
    data2 = line[21:30]
    globals()[f"sat{i+1}"].append(data1)
    globals()[f"sat{i+1}"].append(data2)
    print(globals()[f"sat{i+1}"])

['G19', '21189058.919', '21189058.224', '-23422371.33509', '-1624.826', '21189062.761', '-18232789.09909', '-1266.095']
['G27', '22520607.109', '22520607.184', '-16265062.30809', '-3131.943', '22520615.610', '-12664483.16409', '-2440.473']
['G11', '20960433.392', '20960432.985', '-20236898.56309', ' 2125.069', '20960438.054', '-15753030.25009', ' 1655.895']
['G20', '24599610.601', '24599611.385', ' -4001554.05609', ' 3287.044', '24599619.779', ' -2858090.76509', ' 2561.330']
['G22', '22343853.992', '22343853.460', '-19008308.32409', '-2218.107', '22343856.607', '-14786004.90809', '-1728.398']
['G14', '21370295.018', '21370294.498', '-19001406.30909', ' -529.566', '21370299.873', '-14553929.34709', ' -412.656']
['G32', '21128427.864', '21128427.325', '-19895704.40009', ' 1824.349', '21128433.745', '-15488787.19409', ' 1421.562']
['G01', '22556048.986', '22556050.480', '-15143902.88409', ' 2922.988', '22556058.525', '-11776649.86109', ' 2277.661']
['G31', '       0.000', '25209285.799', 

In [19]:
# PRN C1    P1    L1    D1    P2    L2    D2
# örnek liste görünüm
print(sat1)

['G19', '21189058.919', '21189058.224', '-23422371.33509', '-1624.826', '21189062.761', '-18232789.09909', '-1266.095']


In [20]:
# L1 ve L2 LLI ile SSI göstergeleri ayıklaması
# göstergelerin hepsi 0 ve 9, verileri birbirinden ayırmadım
lli_ssi = []
for i in range(9):
    x = globals()[f"sat{i+1}"][3][-2:]
    globals()[f"sat{i+1}"][3] = globals()[f"sat{i+1}"][3][:-2]
    y = globals()[f"sat{i+1}"][6][-2:]
    globals()[f"sat{i+1}"][6] = globals()[f"sat{i+1}"][6][:-2]    
    lli_ssi.append(globals()[f"sat{i+1}"][0])
    lli_ssi.append(x)
    lli_ssi.append(y)
print(lli_ssi)

['G19', '09', '09', 'G27', '09', '09', 'G11', '09', '09', 'G20', '09', '09', 'G22', '09', '09', 'G14', '09', '09', 'G32', '09', '09', 'G01', '09', '09', 'G31', '09', '09']


In [21]:
# verileri float yap
for i in range(9):
    sat = globals()[f"sat{i+1}"]
    sat[1:] = [float(x) for x in sat[1:]]
    #sat[1:] = [abs(x) for x in sat[1:]] 
    # eksileri yok sayıcakmıyız ????????????????

In [22]:
# PRN C1    P1    L1    D1    P2    L2    D2
# float lli-ssi ayıklandı veriler son kullanıma hazır hali
print(sat1)

['G19', 21189058.919, 21189058.224, -23422371.335, -1624.826, 21189062.761, -18232789.099, -1266.095]


## **3-DIŞARIYA AKTAR**

In [23]:
import pandas as pd

In [24]:
columns = ["Satellite", "C1", "P1", "L1", "D1", "P2", "L2", "D2"]
satellites = [sat1, sat2, sat3, sat4, sat5, sat6, sat7, sat8, sat9]

In [25]:
df = pd.DataFrame(satellites, columns=columns)
df.to_csv("../data/satellites.csv", index=False)